# Price Data Health

Health report for the YF price database (watchlist tickers only). Scores run from 0 to 100: **green** = 95 or above (very healthy), **amber** = 80 to 95 (worth a look), **red** = below 80. Weights and bands are class constants in `pipeline/DataHealth.py`. This notebook only reads data.

In [ ]:
import os, sys
from pathlib import Path
# Work from the project root, wherever this notebook is opened from: every module reads data/ relative to it.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pipeline' / 'watchlist-rules.csv').exists())
os.chdir(root); sys.path.insert(0, str(root))
import pandas as pd
from pipeline.DataHealth import DataHealth

health = DataHealth()
scored = health.score()

## 1. Summary

In [ ]:
for key, value in health.summary(scored).items():
    print(f"{key}: {value}")

## 2. Health map
Sector, then industry, then ticker. Parent tiles take the colour of their worst child. Hover a ticker for its issues.

In [ ]:
health.treemap(scored).show()

## 3. Amber and red tickers
Worst first, with the issues found and a suggested action. To re-fetch a single ticker, use `refresh_ticker('XXX')` on the `YF` class (section 3 of `Data Update.ipynb`) (a dry run by default; pass `apply=True` to write).

In [ ]:
problems = health.problems(scored)
if problems.empty:
    print("No amber or red tickers.")
else:
    pd.set_option('display.max_colwidth', None)
    display(problems)

## 4. Coverage by quarter
Share of expected trading days present, per ticker and quarter, grouped by industry. Blank cells are before a ticker's first date, so history length is visible. Use `health.heatmap(scored, since='2015-01-01')` to zoom in on recent years.

In [ ]:
health.heatmap(scored).show()

# Fundamentals Data Health

Health report for the EDGAR statement files (watchlist tickers only), scored over the last 5 years with the same bands as prices: **green** = use as is, **amber** = needs attention, **red** = exclude from analysis.

Companies that file quarterly are scored on their quarters; foreign filers that only report annually are scored on their annual rows. Tickers the SEC holds no statements for, such as ETFs, are shown as not applicable.

Two things are reported but not marked down: a missing debt figure, and a restated year (the company restated its annual figure after selling a business, so the original quarters no longer sum to it). Both appear in the summary and in a ticker's hover text. Weights are class constants in `pipeline/EdgarHealth.py`. This section only reads data.

In [ ]:
from pipeline.EdgarHealth import EdgarHealth

edgar_health = EdgarHealth()
edgar_scored = edgar_health.score()
for key, value in edgar_health.summary(edgar_scored).items():
    print(f"{key}: {value}")

In [ ]:
edgar_health.treemap(edgar_scored).show()

In [ ]:
# Amber and red tickers, worst first. Most fixes are a tag to add in pipeline/edgar-tags.csv, then EdgarStatements().build().
edgar_problems = edgar_health.problems(edgar_scored)
if edgar_problems.empty:
    print("No amber or red tickers.")
else:
    pd.set_option('display.max_colwidth', None)
    display(edgar_problems)

In [ ]:
# Share of core metrics present per calendar quarter. Red = a quarter that was expected and is missing.
# Blank = nothing expected: annual filers between year ends, and the right-hand edge, where the latest quarter
# has not been reported yet (companies have 40 to 45 days after a quarter ends, so that column fills in gradually).
edgar_health.heatmap(edgar_scored).show()